In [0]:
%run "../0_common/01. env-config"

In [0]:
source_file = f"{landing_folder_path}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"
source_file, table_name

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
races_schema = StructType([
    StructField('season',   IntegerType()),
    StructField("round",    IntegerType()),
    StructField("url",      StringType()),
    StructField("raceName", StringType()),
    StructField("date",     DateType()),
    StructField("circuitId", StringType())
])

In [0]:
df = (
    spark.read
    .format('csv')
    .option("header","true")
    .schema(races_schema)
    .load(source_file)
)

In [0]:
display(df.head(5))

In [0]:
import pyspark.sql.functions as F
df = (
    df
    .withColumn("ingestion_timestamp",F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

In [0]:
display(df.head(5))

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
display(spark.table(table_name))